# Cosine Similarity and Angle between Decoder Weights (RS, OF, Depth)

This notebook loads the continuous decoder weight parquets for treadmill sessions,
computes the cosine similarity and angle (in degrees) for all pairwise combinations of the three decoders
(Running Speed `RS`, Optic Flow `OF`, and Depth) for each session, and plots the similarity and corresponding
angles against nominal recording depth.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from cottage_analysis.summary_analysis import get_session_list, load_project_neurons

# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {"PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"}

In [ ]:
# Load data for each project separately
df_ccyp = load_project_neurons("ccyp_l5_3d_vision", session_to_exclude=SESSIONS_TO_EXCLUDE.keys())
df_colasa = load_project_neurons("colasa_3d-vision_revisions", session_to_exclude=SESSIONS_TO_EXCLUDE.keys())

# Add project identifiers
df_ccyp["project"] = "ccyp_l5_3d_vision"
df_colasa["project"] = "colasa_3d-vision_revisions"

# Combine them
df_all = pd.concat([df_ccyp, df_colasa], ignore_index=True)
print(f"Loaded a total of {len(df_all)} neurons across both projects.")

In [ ]:
# Group by session and project/nominal depth, then compute cosine similarity
cut_off = 350
results = []
for (session, project, nominal_depth), group in df_all.groupby(["session", "project", "nominal_depth"]):
    # Drop rows containing NaN weights (e.g. if some neurons had NaN values and were excluded)
    group_valid = group.dropna(subset=["ridge_weight_mean_RS_stim_closedloop", "ridge_weight_mean_OF_stim_closedloop", "ridge_weight_mean_depth_closedloop"])
    
    if len(group_valid) == 0:
        print(f"Skipping {session}: no valid weights.")
        continue
        
    w_RS = group_valid["ridge_weight_mean_RS_stim_closedloop"].values
    w_OF = group_valid["ridge_weight_mean_OF_stim_closedloop"].values
    w_depth = group_valid["ridge_weight_mean_depth_closedloop"].values
    
    # Calculate Cosine Similarity for all combinations
    cos_sim_rs_of = np.dot(w_RS, w_OF) / (np.linalg.norm(w_RS) * np.linalg.norm(w_OF))
    angle_rs_of = np.degrees(np.arccos(np.clip(cos_sim_rs_of, -1.0, 1.0)))
    
    cos_sim_rs_depth = np.dot(w_RS, w_depth) / (np.linalg.norm(w_RS) * np.linalg.norm(w_depth))
    angle_rs_depth = np.degrees(np.arccos(np.clip(cos_sim_rs_depth, -1.0, 1.0)))
    
    cos_sim_of_depth = np.dot(w_OF, w_depth) / (np.linalg.norm(w_OF) * np.linalg.norm(w_depth))
    angle_of_depth = np.degrees(np.arccos(np.clip(cos_sim_of_depth, -1.0, 1.0)))
    
    results.append({
        "session": session,
        "project": project,
        "nominal_depth": nominal_depth,
        "layer": 'Layer 2/3' if nominal_depth <= cut_off else 'Layer 5',
        "n_neurons": len(group_valid),
        "cos_sim_rs_of": cos_sim_rs_of,
        "angle_rs_of": angle_rs_of,
        "cos_sim_rs_depth": cos_sim_rs_depth,
        "angle_rs_depth": angle_rs_depth,
        "cos_sim_of_depth": cos_sim_of_depth,
        "angle_of_depth": angle_of_depth
    })

df_similarity = pd.DataFrame(results)
print("Computed cosine similarity for all sessions:")
df_similarity_sorted = df_similarity.sort_values(by="nominal_depth")
pd.set_option('display.max_rows', 100)
df_similarity_sorted

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)

# Row 0: Cosine Similarities
sns.scatterplot(data=df_similarity, x="nominal_depth", y="cos_sim_rs_of",  s=100, ax=axes[0, 0], palette="Set1")
axes[0, 0].axhline(0, color="gray", linestyle="--")
axes[0, 0].set_ylabel("Cosine Similarity (RS vs OF)")
axes[0, 0].set_title("RS vs OF Cosine Similarity")

sns.scatterplot(data=df_similarity, x="nominal_depth", y="cos_sim_rs_depth", s=100, ax=axes[0, 1], palette="Set1")
axes[0, 1].axhline(0, color="gray", linestyle="--")
axes[0, 1].set_ylabel("Cosine Similarity (RS vs Depth)")
axes[0, 1].set_title("RS vs Depth Cosine Similarity")

sns.scatterplot(data=df_similarity, x="nominal_depth", y="cos_sim_of_depth", s=100, ax=axes[0, 2], palette="Set1")
axes[0, 2].axhline(0, color="gray", linestyle="--")
axes[0, 2].set_ylabel("Cosine Similarity (OF vs Depth)")
axes[0, 2].set_title("OF vs Depth Cosine Similarity")

# Row 1: Angles
sns.scatterplot(data=df_similarity, x="nominal_depth", y="angle_rs_of", s=100, ax=axes[1, 0], palette="Set1")
axes[1, 0].axhline(90, color="gray", linestyle="--")
axes[1, 0].set_xlabel("Cortical Depth (μm)")
axes[1, 0].set_ylabel("Angle (degrees) (RS vs OF)")
axes[1, 0].set_title("RS vs OF Angle")

sns.scatterplot(data=df_similarity, x="nominal_depth", y="angle_rs_depth",s=100, ax=axes[1, 1], palette="Set1")
axes[1, 1].axhline(90, color="gray", linestyle="--")
axes[1, 1].set_xlabel("Cortical Depth (μm)")
axes[1, 1].set_ylabel("Angle (degrees) (RS vs Depth)")
axes[1, 1].set_title("RS vs Depth Angle")

sns.scatterplot(data=df_similarity, x="nominal_depth", y="angle_of_depth", s=100, ax=axes[1, 2], palette="Set1")
axes[1, 2].axhline(90, color="gray", linestyle="--")
axes[1, 2].set_xlabel("Cortical Depth (μm)")
axes[1, 2].set_ylabel("Angle (degrees) (OF vs Depth)")
axes[1, 2].set_title("OF vs Depth Angle")

plt.tight_layout()
sns.despine()

# Save the figure
fig_path = Path("rs_of_cosine_similarity_vs_depth.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path.resolve()}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean, publication-ready style
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(7, 5))

# Define colors for each comparison
colors = {
    "RS vs OF": "#1f77b4",     # Blue
    "RS vs Depth": "#2ca02c",  # Green
    "OF vs Depth": "#d62728"   # Red
}

# Plot line + scatter for each comparison using Cosine Similarity columns
sns.lineplot(data=df_similarity, x="nominal_depth", y="cos_sim_rs_of", label="RS vs OF", 
             color=colors["RS vs OF"], marker="o", ls='', markersize=8, linewidth=2, ax=ax)
sns.lineplot(data=df_similarity, x="nominal_depth", y="cos_sim_rs_depth", label="RS vs Depth", 
             color=colors["RS vs Depth"], marker="s", ls='', markersize=8, linewidth=2, ax=ax)
sns.lineplot(data=df_similarity, x="nominal_depth", y="cos_sim_of_depth", label="OF vs Depth", 
             color=colors["OF vs Depth"], marker="^", ls='', markersize=8, linewidth=2, ax=ax)

# Reference lines at 0 (orthogonal) and -1 (opposite)
ax.axhline(0, color="gray", linestyle="--", alpha=0.7, label="0 (Orthogonal)")
ax.axhline(-1, color="darkgray", linestyle="-.", alpha=0.7, label="-1 (Opposite)")
ax.axhline(1, color="lightgray", linestyle=":", alpha=0.5)

# Styling
ax.set_xlabel("Cortical Depth (μm)", fontsize=11, fontweight="bold")
ax.set_ylabel("Cosine Similarity", fontsize=11, fontweight="bold")
ax.set_title("Cosine Similarity Between Vectors Across Cortical Depth", fontsize=12, fontweight="bold", pad=12)
ax.set_ylim(-1.1, 1.1)
ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])

# Move legend to a clean spot
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=9, loc="upper right")

sns.despine(trim=True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Fix 1: Keep 'layer' as an identifier variable instead of melting it
df_melted = df_similarity.melt(
    id_vars=["nominal_depth", "layer"],
    value_vars=["angle_rs_of", "angle_rs_depth", "angle_of_depth"],
    var_name="Comparison",
    value_name="Angle"
)

# Rename comparisons for clean labels
df_melted["Comparison"] = df_melted["Comparison"].map({
    "angle_rs_of": "RS vs OF",
    "angle_rs_depth": "RS vs Depth",
    "angle_of_depth": "OF vs Depth"
})

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(7, 5))

# Reference lines at 90 and 180 degrees
ax.axhline(90, color="gray", linestyle="--", alpha=0.6, zorder=1)
ax.axhline(180, color="gray", linestyle="-.", alpha=0.6, zorder=1)

# Boxplot grouped by layer
sns.boxplot(
    data=df_melted, 
    x="Comparison", 
    y="Angle", 
    hue="layer", 
    palette="Set2", 
    width=0.6, 
    showfliers=False, 
    ax=ax, 
    zorder=2
)

# Fix 2: Add hue, dodge=True, and hide extra legend entries for the stripplot
sns.stripplot(
    data=df_melted, 
    x="Comparison", 
    y="Angle", 
    hue="layer",
    dodge=True,
    color="black",
    alpha=0.5, 
    size=5, 
    jitter=0.15, 
    ax=ax, 
    zorder=3,
    legend=False  # Avoids duplicate legend entries
)

# Styling
ax.set_ylabel("Angle (degrees)", fontsize=11, fontweight="bold")
ax.set_xlabel("", fontsize=11)
ax.set_title("Distribution of Vector Angles by Layer", fontsize=12, fontweight="bold", pad=12)
ax.set_ylim(0, 200)
ax.set_yticks([0, 45, 90, 135, 180])

# Move legend to a neat place
ax.legend(title="Layer", frameon=True, facecolor="white", edgecolor="none")

sns.despine(trim=True)
plt.tight_layout()
plt.show()


In [ ]:
# 1. Drop rows with missing weights in any of the three target conditions
df_weights = df_all.dropna(subset=[
    "ridge_weight_mean_RS_stim_closedloop", 
    "ridge_weight_mean_OF_stim_closedloop", 
    "ridge_weight_mean_depth_closedloop"
]).copy()

# 2. Rename columns to shorter names for easier plotting/melting
df_weights = df_weights.rename(columns={
    "ridge_weight_mean_RS_stim_closedloop": "w_rs",
    "ridge_weight_mean_OF_stim_closedloop": "w_of",
    "ridge_weight_mean_depth_closedloop": "w_depth"
})

# 3. Add the 'layer' column based on your nominal depth cutoff (350 um)
cut_off = 350
df_weights["layer"] = df_weights["nominal_depth"].apply(
    lambda d: 'Layer 2/3' if d <= cut_off else 'Layer 5'
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Compute robust color limits (vmin/vmax) using the 95th percentile
# to avoid outliers making the plot look pale
vmax = np.percentile(df_weights[['w_rs', 'w_of', 'w_depth']].abs(), 95)
vmin = -vmax

# 2. Generate a separate clustered heatmap for each layer
for layer in ["Layer 2/3", "Layer 5"]:
    # Filter for the current layer
    df_layer = df_weights[df_weights["layer"] == layer][['w_rs', 'w_of', 'w_depth']]
    
    if len(df_layer) == 0:
        print(f"No neurons found for {layer}")
        continue
        
    g = sns.clustermap(
        df_layer,
        cmap="RdBu_r",          # Red = positive, Blue = negative
        center=0,
        vmin=vmin,              # Set lower clim
        vmax=vmax,              # Set upper clim
        col_cluster=False,      # Keep column order: RS, OF, Depth
        row_cluster=True,       # Cluster the neurons
        yticklabels=False,      # Hide individual neuron indexes
        cbar_kws={'label': 'Weight Value'},
        figsize=(5, 7)
    )
    g.fig.suptitle(f"Clustering of {layer} Neuron Weights", y=1.02, fontweight="bold")
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Corner plot showing 2D distributions of weights
g = sns.pairplot(
    df_weights[['w_rs', 'w_of', 'w_depth', 'layer']],
    vars=['w_rs', 'w_of', 'w_depth'],    
    kind="scatter",
    hue='layer',
    diag_kind="hist",
    diag_kws={'stat': 'density', 'common_norm': False, 'alpha': 0.8, 'element': 'step', 'fill': False},
    plot_kws={'alpha': 0.3, 'edgecolor': 'none', 's': 5},
    corner=True
)

# Add dashed orthogonal lines at x=0 and y=0 to highlight axis alignment
for i, ax_row in enumerate(g.axes):
    for j, ax in enumerate(ax_row):
        if ax is not None:
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
            
g.fig.suptitle("Pairwise Weight Relationships", y=1.02, fontweight="bold")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_all_sessions_by_layer_3d(df_all, normalize=True):
    """
    Plots vectors for all sessions split by Layer 2/3 (left) and Layer 5 (right).
    """
    cut_off = 350
    
    # Categorize layers if 'layer' column is not in df_all
    if "layer" not in df_all.columns:
        df_all = df_all.copy()
        df_all["layer"] = df_all["nominal_depth"].apply(
            lambda d: 'Layer 2/3' if d <= cut_off else 'Layer 5'
        )
        
    # Set up figure with 2 subplots side-by-side
    fig = plt.figure(figsize=(15, 7.5))
    ax_l23 = fig.add_subplot(121, projection='3d')
    ax_l5 = fig.add_subplot(122, projection='3d')
    
    axes_dict = {'Layer 2/3': ax_l23, 'Layer 5': ax_l5}
    colors = ["#1f77b4", "#d62728", "#2ca02c"] # Blue (RS), Red (OF), Green (Depth)
    labels = ["RS", "OF", "Depth"]
    
    for layer, ax in axes_dict.items():
        # Plot origin dot
        ax.scatter(0, 0, 0, color='black', s=40, zorder=5)
        
        # Get unique sessions for this layer
        layer_sessions = df_all[df_all["layer"] == layer]["session"].unique()
        
        legend_added = False
        for session in layer_sessions:
            group = df_all[df_all["session"] == session]
            group_valid = group.dropna(subset=[
                "ridge_weight_mean_RS_stim_closedloop", 
                "ridge_weight_mean_OF_stim_closedloop", 
                "ridge_weight_mean_depth_closedloop"
            ])
            
            if len(group_valid) == 0:
                continue
                
            w_RS = group_valid["ridge_weight_mean_RS_stim_closedloop"].values
            w_OF = group_valid["ridge_weight_mean_OF_stim_closedloop"].values
            w_depth = group_valid["ridge_weight_mean_depth_closedloop"].values
            
            norm_RS = np.linalg.norm(w_RS)
            norm_OF = np.linalg.norm(w_OF)
            norm_depth = np.linalg.norm(w_depth)
            
            u_RS = w_RS / norm_RS
            u_OF = w_OF / norm_OF
            u_depth = w_depth / norm_depth
            
            # Compute Gram matrix
            G = np.array([
                [1.0, np.dot(u_RS, u_OF), np.dot(u_RS, u_depth)],
                [np.dot(u_OF, u_RS), 1.0, np.dot(u_OF, u_depth)],
                [np.dot(u_depth, u_RS), np.dot(u_depth, u_OF), 1.0]
            ])
            G += np.eye(3) * 1e-9 # stability
            
            L = np.linalg.cholesky(G)
            
            if not normalize:
                L[0, :] *= norm_RS
                L[1, :] *= norm_OF
                L[2, :] *= norm_depth
                
            v_RS, v_OF, v_depth = L[0, :], L[1, :], L[2, :]
            vectors = [v_RS, v_OF, v_depth]
            
            # Draw vectors as quivers
            for v, color, label in zip(vectors, colors, labels):
                # Avoid duplicate labels in the legend
                lbl = label if not legend_added else None
                ax.quiver(
                    0, 0, 0, v[0], v[1], v[2], 
                    color=color, 
                    linewidth=2.5, 
                    alpha=0.5, 
                    label=lbl
                )
            legend_added = True
            
        # Limits and formatting
        lim = 1.2 if normalize else max(df_all["nominal_depth"]) * 0.005 # scaling placeholder if raw
        ax.set_xlim([-lim, lim])
        ax.set_ylim([-lim, lim])
        ax.set_zlim([-lim, lim])
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(f"{layer} (n={len(layer_sessions)} sessions)", fontweight='bold', fontsize=12)
        ax.legend(loc="upper left")
        
    plt.suptitle("Vector Alignment across Sessions by Layer", fontsize=14, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.show()

# Call the function
plot_all_sessions_by_layer_3d(df_all, normalize=True)
